In [1]:
import pandas as pd

books = pd.read_csv("../data/books_cleaned.csv")

In [2]:
books["categories"].value_counts().reset_index()

,categories,count
0,Exhibitions,11
1,American literature,11
2,Children's fiction,8
3,"Fiction, romance, general",7
4,Literature,6
...,...,...
4695,Baking;Cooking (Natural foods);Fermented foods...,1
4696,"Literature;Comic books, strips;Childhood and y...",1
4697,Religion;Philosophy;Liberty;Islam;Muslims;Reli...,1
4698,"Byzantine empire, history;Macedonia, history;H...",1


In [3]:
# get rid of categories with less than 50 books
books["categories"].value_counts().reset_index().query("count > 50")

,categories,count


In [4]:
import re

# OL puts many subjects into one ';'-joined string so scan for keywords. 
# "nonfiction" contains "fiction", so nonfiction tested before fiction.

# genre hints consulted when no explicit fiction/nonfiction label is present.
FICTION_HINTS = (
    "fantasy", "romance", "thriller", "mystery", "horror", "science fiction",
    "short stories", "fairy tale", "graphic novel", "comic", "detective",
    "adventure", "poetry", "drama", "novel",
)
NONFICTION_HINTS = (
    "biography", "autobiography", "history", "philosophy", "religion", "self-help",
    "cooking", "travel", "business", "psychology", "memoir", "true crime", "essays",
    "science", "reference", "health",
)

def simplify_categories(cats):
    if not isinstance(cats, str):
        return None
    c = cats.lower()

    # children's books collapse to one bucket: the fiction/nonfiction sub-split was sparse
    # (Children's Nonfiction ~1%) and noisy, and the juvenile signal is far more reliable
    # than that sub-decision. Checked first, so we don't even need to resolve fiction kind.
    if ("juvenile" in c) or ("children" in c):
        return "Children's"

    if re.search(r"non-?fiction", c):              # 1: explicit labels
        return "Nonfiction"
    if re.search(r"\bfiction\b", c):
        return "Fiction"
    if any(h in c for h in FICTION_HINTS):         # 2: genre hints
        return "Fiction"
    if any(h in c for h in NONFICTION_HINTS):
        return "Nonfiction"
    return None                                    # leave for the BART backfill below

books["simple_categories"] = books["categories"].apply(simplify_categories)
books["simple_categories"].value_counts(dropna=False)

simple_categories
Nonfiction    1827
NaN           1501
Fiction       1117
Children's     555
Name: count, dtype: int64

In [5]:
books

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories
0,9781804510407,Other Side of the Wire,Ralph J. Whitehead,"Germany. Heer;Germany. Heer. Reserve Corps, 14...","""By focusing on one of the principal German fo...",NaN,2022,NaN,NaN,Other Side of the Wire: Volume 2 - the Battle ...,"9781804510407 ""By focusing on one of the princ...",Nonfiction
1,9781108663229,African Literature and the CIA,Caroline Davis,Written communication;African literature;CIA;U...,"""During the period of decolonisation in Africa...",NaN,2020,NaN,NaN,African Literature and the CIA: Networks of Au...,"9781108663229 ""During the period of decolonisa...",Nonfiction
2,9780367418489,Interior Provocations,Anca I. Lasc;Pratt Institute Staff,Architecture;Interior architecture;Congresses;...,"""Interior Provocations: History, Theory, and P...",NaN,2020,NaN,NaN,Interior Provocations,"9780367418489 ""Interior Provocations: History,...",NaN
3,9781800326514,An Unfortunate Christmas Murder,Hannah Hendy,English literature,"‘Tis the season for gold, frankincense and mur...",https://covers.openlibrary.org/b/id/15179740-L...,2022,NaN,NaN,An Unfortunate Christmas Murder,"9781800326514 ‘Tis the season for gold, franki...",NaN
4,9781108485852,Legitimacy of Unseen Actors in International A...,Freya Baetens,Arbitration (international law);Jurisdiction (...,"""'Unseen actors' are vital to the functioning ...",NaN,2019,NaN,496.0,Legitimacy of Unseen Actors in International A...,"9781108485852 ""'Unseen actors' are vital to th...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,9781952223174,"Why, As a Muslim, I Support Liberty",Mustafa Akyol,Religion;Philosophy;Liberty;Islam;Muslims;Reli...,"Islam, the second largest religion in the worl...",NaN,2021,NaN,192.0,"Why, As a Muslim, I Support Liberty","9781952223174 Islam, the second largest religi...",Nonfiction
4996,9781532147555,The Horror at Happy Landings,Robert Lawrence Stine;Kelly Matthews;Nichole M...,NaN,Two Martians unexpectedly land on Earth and ha...,https://covers.openlibrary.org/b/id/14408331-L...,2020,NaN,NaN,The Horror at Happy Landings: Just Beyond Volu...,9781532147555 Two Martians unexpectedly land o...,NaN
4997,9789004278783,The Blinded State,Mitko B. Panov,"Byzantine empire, history;Macedonia, history;H...","""This book is a revisionist account of Samuel'...",NaN,2019,NaN,478.0,The Blinded State,"9789004278783 ""This book is a revisionist acco...",Nonfiction
4998,9781784753474,Walls,Hollie Overton,"Abused women;Murder;Fiction;Fiction, suspense;...","""For fans of The Girl on the Train, The Walls ...",NaN,2018,NaN,416.0,Walls,"9781784753474 ""For fans of The Girl on the Tra...",Fiction


In [6]:
books[~(books["simple_categories"].isna())]

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories
0,9781804510407,Other Side of the Wire,Ralph J. Whitehead,"Germany. Heer;Germany. Heer. Reserve Corps, 14...","""By focusing on one of the principal German fo...",NaN,2022,NaN,NaN,Other Side of the Wire: Volume 2 - the Battle ...,"9781804510407 ""By focusing on one of the princ...",Nonfiction
1,9781108663229,African Literature and the CIA,Caroline Davis,Written communication;African literature;CIA;U...,"""During the period of decolonisation in Africa...",NaN,2020,NaN,NaN,African Literature and the CIA: Networks of Au...,"9781108663229 ""During the period of decolonisa...",Nonfiction
6,9781455591183,Engine 2 Cookbook,Rip Esselstyn;Jane Esselstyn,Health;Vegetarian cooking;Home economics,With this highly anticipated follow-up to the ...,NaN,2021,NaN,288.0,"Engine 2 Cookbook: More Than 130 Lip-Smacking,...",9781455591183 With this highly anticipated fol...,Nonfiction
7,9780821423547,The Wolf at Number 4,Ayo Tamakloe-Garr,"Fiction, general;Africa, fiction;FICTION / Gen...","""Desire Mensah, a disgraced school teacher in ...",https://covers.openlibrary.org/b/id/8810722-L.jpg,2018,NaN,180.0,The Wolf at Number 4: A Novel,"9780821423547 ""Desire Mensah, a disgraced scho...",Fiction
8,9781925760804,Travelling companions,Antoni Jach,Australian fiction;Strangers;Fiction;Storytell...,Solitary travellers and a couple encounter Nin...,NaN,2021,NaN,408.0,Travelling companions,9781925760804 Solitary travellers and a couple...,Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...
4993,9781841884066,Healthy Baking,Jordan Bourke,Baking;Cooking (Natural foods);Fermented foods...,This beautiful full-colour cookbook features n...,NaN,2019,NaN,280.0,"Healthy Baking: Nourishing Breads, Wholesome C...",9781841884066 This beautiful full-colour cookb...,Nonfiction
4994,9781770463592,King of King Court,Travis Dandro,"Literature;Comic books, strips;Childhood and y...","""From a child's-eye view, Travis Dandro recoun...",https://covers.openlibrary.org/b/id/10118354-L...,2019,NaN,464.0,King of King Court,"9781770463592 ""From a child's-eye view, Travis...",Children's
4995,9781952223174,"Why, As a Muslim, I Support Liberty",Mustafa Akyol,Religion;Philosophy;Liberty;Islam;Muslims;Reli...,"Islam, the second largest religion in the worl...",NaN,2021,NaN,192.0,"Why, As a Muslim, I Support Liberty","9781952223174 Islam, the second largest religi...",Nonfiction
4997,9789004278783,The Blinded State,Mitko B. Panov,"Byzantine empire, history;Macedonia, history;H...","""This book is a revisionist account of Samuel'...",NaN,2019,NaN,478.0,The Blinded State,"9789004278783 ""This book is a revisionist acco...",Nonfiction


In [7]:
from transformers import pipeline

fiction_categories = ["Fiction", "Nonfiction"]
pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device="mps")
# mps is the Apple Mac specific GPU

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [8]:
sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[0]
# resetting the index ensure indexes correlate to the classified books not the original database

In [9]:
pipe(sequence, fiction_categories)
# returns probability that the book is within each category

{'sequence': '"Desire Mensah, a disgraced school teacher in her thirties, sees moving to sleepy little Cape Coast, Ghana, as her chance to get away from a shameful secret not buried deeply enough. And maybe, just maybe, she will find the love she craves and the husband her mother craves for her.But in Cape Coast, the past isn\'t dead. It isn\'t even past. That\'s the kind of thing Wolfgang "Wolf" Ofori would say. Everyone says the eleven-year-old is a genius--eccentric though he is--and is bound to win Wonderkids, a quiz competition ordinarily for high school students. Wolf and Desire form a strange friendship, even as their mutual understanding both precipitates and reinforces each\'s downfall. Before long, their struggles to exist in a world that dehumanizes and then throws the first stone whips up a perfect storm, with deadly consequences.Debut novelist Ayo Tamakloe-Garr drew inspiration from works such as A Streetcar Named Desire and Frankenstein to create The Wolf at Number 4, a s

In [10]:
import numpy as np

max_index = np.argmax(pipe(sequence, fiction_categories)["scores"])
max_label = pipe(sequence, fiction_categories)["labels"][max_index]
max_label

'Fiction'

In [11]:
def generate_predictions(sequence, categories):
    predictions = pipe(sequence, categories)
    max_index = np.argmax(predictions["scores"]) # yields the index of the highest probability
    max_label = predictions["labels"][max_index]
    return max_label

In [12]:
# Testing how good the model is using a sizable sample
from tqdm import tqdm
# tqdm is a library used to add progress bars to loops

actual_cats = []
predicted_cats = []

for i in tqdm(range(0, 100)):
    sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Fiction"]

100%|██████████| 100/100 [00:16<00:00,  6.00it/s]


In [13]:
for i in tqdm(range(0, 100)):
    sequence = books.loc[books["simple_categories"] == "Nonfiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Nonfiction"]

100%|██████████| 100/100 [00:16<00:00,  6.06it/s]


In [14]:
predictions_df = pd.DataFrame({"actual_categories": actual_cats, "predicted_categories": predicted_cats})
predictions_df

,actual_categories,predicted_categories
0,Fiction,Fiction
1,Fiction,Nonfiction
2,Fiction,Fiction
3,Fiction,Fiction
4,Fiction,Nonfiction
...,...,...
195,Nonfiction,Fiction
196,Nonfiction,Nonfiction
197,Nonfiction,Nonfiction
198,Nonfiction,Nonfiction


In [15]:
predictions_df["correct_prediction"] = (
    np.where(predictions_df["actual_categories"] == predictions_df["predicted_categories"], 1, 0)
)

In [16]:
# Percentage accuracy of the BART zero-shot Fiction/Nonfiction backfill on the 200-book labelled sample.
# TODO: could be improved in the future
predictions_df["correct_prediction"].sum() / len(predictions_df)

np.float64(0.75)

In [17]:
# make a subset of the database with the simple category missing
isbns = []
predicted_cats = []

missing_cats = books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)

In [18]:
for i in tqdm(range(0, len(missing_cats))):
    sequence = missing_cats["description"][i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    isbns += [missing_cats["isbn13"][i]]

100%|██████████| 1501/1501 [03:23<00:00,  7.36it/s]


In [19]:
missing_predicted_df = pd.DataFrame({"isbn13": isbns, "predicted_categories": predicted_cats})

In [20]:
missing_predicted_df

,isbn13,predicted_categories
0,9780367418489,Nonfiction
1,9781800326514,Fiction
2,9781108485852,Nonfiction
3,9781522556602,Nonfiction
4,9789563988406,Nonfiction
...,...,...
1496,9781000319682,Nonfiction
1497,9781097462919,Nonfiction
1498,9781350177789,Nonfiction
1499,9781532147555,Fiction


In [21]:
books = pd.merge(books, missing_predicted_df, on="isbn13", how="left") 
books["simple_categories"] = np.where(books["simple_categories"].isna(), books["predicted_categories"], books["simple_categories"])
# merge into books - when category is missing use predicted, if simple category there then use that
books = books.drop(columns = ["predicted_categories"])

In [22]:
books

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories
0,9781804510407,Other Side of the Wire,Ralph J. Whitehead,"Germany. Heer;Germany. Heer. Reserve Corps, 14...","""By focusing on one of the principal German fo...",NaN,2022,NaN,NaN,Other Side of the Wire: Volume 2 - the Battle ...,"9781804510407 ""By focusing on one of the princ...",Nonfiction
1,9781108663229,African Literature and the CIA,Caroline Davis,Written communication;African literature;CIA;U...,"""During the period of decolonisation in Africa...",NaN,2020,NaN,NaN,African Literature and the CIA: Networks of Au...,"9781108663229 ""During the period of decolonisa...",Nonfiction
2,9780367418489,Interior Provocations,Anca I. Lasc;Pratt Institute Staff,Architecture;Interior architecture;Congresses;...,"""Interior Provocations: History, Theory, and P...",NaN,2020,NaN,NaN,Interior Provocations,"9780367418489 ""Interior Provocations: History,...",Nonfiction
3,9781800326514,An Unfortunate Christmas Murder,Hannah Hendy,English literature,"‘Tis the season for gold, frankincense and mur...",https://covers.openlibrary.org/b/id/15179740-L...,2022,NaN,NaN,An Unfortunate Christmas Murder,"9781800326514 ‘Tis the season for gold, franki...",Fiction
4,9781108485852,Legitimacy of Unseen Actors in International A...,Freya Baetens,Arbitration (international law);Jurisdiction (...,"""'Unseen actors' are vital to the functioning ...",NaN,2019,NaN,496.0,Legitimacy of Unseen Actors in International A...,"9781108485852 ""'Unseen actors' are vital to th...",Nonfiction
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,9781952223174,"Why, As a Muslim, I Support Liberty",Mustafa Akyol,Religion;Philosophy;Liberty;Islam;Muslims;Reli...,"Islam, the second largest religion in the worl...",NaN,2021,NaN,192.0,"Why, As a Muslim, I Support Liberty","9781952223174 Islam, the second largest religi...",Nonfiction
4996,9781532147555,The Horror at Happy Landings,Robert Lawrence Stine;Kelly Matthews;Nichole M...,NaN,Two Martians unexpectedly land on Earth and ha...,https://covers.openlibrary.org/b/id/14408331-L...,2020,NaN,NaN,The Horror at Happy Landings: Just Beyond Volu...,9781532147555 Two Martians unexpectedly land o...,Fiction
4997,9789004278783,The Blinded State,Mitko B. Panov,"Byzantine empire, history;Macedonia, history;H...","""This book is a revisionist account of Samuel'...",NaN,2019,NaN,478.0,The Blinded State,"9789004278783 ""This book is a revisionist acco...",Nonfiction
4998,9781784753474,Walls,Hollie Overton,"Abused women;Murder;Fiction;Fiction, suspense;...","""For fans of The Girl on the Train, The Walls ...",NaN,2018,NaN,416.0,Walls,"9781784753474 ""For fans of The Girl on the Tra...",Fiction


In [23]:
books.to_csv("../data/books_with_categories.csv", index=False)